In [2]:
pip install xarray

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   -------------- ------------------------- 0.5/1.4 MB 27.4 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 5.2 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
%pip install netCDF4 h5netcdf

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/21.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/21.3 MB ? eta -:--:--
    --------------------------------------- 0.5/21.3 MB 2.5 MB/s eta 0:00:09
   - -------------------------------------- 1.0/21.3 MB 2.5 MB/s eta 0:00:09
   --- ------------------------------------ 1.8/21.3 MB 3.1 MB/s eta 0:00:07
   ----- ---------------------------------- 2.9/21.3 MB 3.6 MB/s eta 0:00:06
   ----- ---------------------------------- 3.1/21.3 MB 3.5 MB/s eta 0:00:06
   ------ --------------------------------- 3.7/21.3 MB 3.0 MB/s eta 0:00:06
   --------- ------------------------------ 5.0/21.3 MB 3.4 MB/s eta 0:00:05
   ------------ --------------------------- 6.8/21.3 MB 4.1 MB/s eta 0:00:04
   --------------- ------------------------ 8.4/21.3 MB 4.6 MB/s eta 0:00:03
   ------------------ --------------------- 10.0/21.3 MB 5.0 MB/s eta 0:00:03
   --------


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# ============================================================
# CELL 1 — Setup
# ============================================================

from pathlib import Path
import numpy as np
import xarray as xr

ROOT = Path.cwd()
while not (ROOT / "data" / "raw" / "era5").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

ERA5_DIR = ROOT / "data" / "raw" / "era5"
OUTPUT = ROOT / "data" / "intermediate" / "era5_merged_2021_2025.nc"

YEARS = range(2021, 2026)

print("ERA5 input :", ERA5_DIR)
print("Output     :", OUTPUT)

ERA5 input : c:\Users\udish\College\MajorProjectSem7\Implementation\data\raw\era5
Output     : c:\Users\udish\College\MajorProjectSem7\Implementation\data\intermediate\era5_merged_2021_2025.nc


In [2]:
# ============================================================
# CELL 2 — File Inventory
# ============================================================

files = {
    year: {
        "instant": ERA5_DIR / str(year) / f"era5_instant_{year}.nc",
        "accum": ERA5_DIR / str(year) / f"era5_accum_{year}.nc",
        "blh": ERA5_DIR / str(year) / f"era5_blh_{year}.nc"
    }
    for year in YEARS
}

for year, f in files.items():
    print(
        year,
        "| instant:", f["instant"].exists(),
        "| accum:", f["accum"].exists(),
        "| blh:", f["blh"].exists()
    )

assert all(
    p.exists()
    for year in files.values()
    for p in year.values()
), "One or more ERA5 files are missing."

print("PASS — 15 ERA5 files found.")

2021 | instant: True | accum: True | blh: True
2022 | instant: True | accum: True | blh: True
2023 | instant: True | accum: True | blh: True
2024 | instant: True | accum: True | blh: True
2025 | instant: True | accum: True | blh: True
PASS — 15 ERA5 files found.


In [3]:
# ============================================================
# CELL 3 — Merge One Year
# ============================================================

def merge_one_year(f, year):
    ds_i = xr.open_dataset(f["instant"], engine="netcdf4")
    ds_a = xr.open_dataset(f["accum"], engine="netcdf4")
    ds_b = xr.open_dataset(f["blh"], engine="netcdf4")[["blh"]]

    ds_i = ds_i.drop_vars("expver", errors="ignore")
    ds_a = ds_a.drop_vars("expver", errors="ignore")
    ds_b = ds_b.drop_vars("expver", errors="ignore")

    if not np.array_equal(
        ds_b.valid_time.values,
        ds_i.valid_time.values
    ):
        ds_b = ds_b.reindex(
            valid_time=ds_i.valid_time,
            method="nearest"
        )

    merged = xr.merge([ds_i, ds_a, ds_b])

    print(f"{year}: {len(merged.valid_time)} timesteps")
    return merged

In [4]:
# ============================================================
# CELL 4 — Build Full ERA5 Dataset
# ============================================================

yearly = [
    merge_one_year(files[year], year)
    for year in YEARS
]

era5_full = xr.concat(
    yearly,
    dim="valid_time"
).sortby("valid_time")

_, unique_idx = np.unique(
    era5_full.valid_time.values,
    return_index=True
)

if len(unique_idx) < len(era5_full.valid_time):
    era5_full = era5_full.isel(
        valid_time=sorted(unique_idx)
    )

print("\nFinal dataset:")
print(dict(era5_full.sizes))
print(
    "Time range:",
    era5_full.valid_time.values[0],
    "→",
    era5_full.valid_time.values[-1]
)

C:\Users\udish\AppData\Local\Temp\ipykernel_23104\3920530274.py:23: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'latitude' ('latitude',) The recommendation is to set join explicitly for this case.
  merged = xr.merge([ds_i, ds_a, ds_b])
C:\Users\udish\AppData\Local\Temp\ipykernel_23104\3920530274.py:23: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  merged = xr.merge([ds_i, ds_a, ds_b])


2021: 2920 timesteps


C:\Users\udish\AppData\Local\Temp\ipykernel_23104\3920530274.py:23: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  merged = xr.merge([ds_i, ds_a, ds_b])


2022: 2920 timesteps


C:\Users\udish\AppData\Local\Temp\ipykernel_23104\3920530274.py:23: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'latitude' ('latitude',) The recommendation is to set join explicitly for this case.
  merged = xr.merge([ds_i, ds_a, ds_b])
C:\Users\udish\AppData\Local\Temp\ipykernel_23104\3920530274.py:23: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  merged = xr.merge([ds_i, ds_a, ds_b])


2023: 2920 timesteps


C:\Users\udish\AppData\Local\Temp\ipykernel_23104\3920530274.py:23: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'latitude' ('latitude',) The recommendation is to set join explicitly for this case.
  merged = xr.merge([ds_i, ds_a, ds_b])
C:\Users\udish\AppData\Local\Temp\ipykernel_23104\3920530274.py:23: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  merged = xr.merge([ds_i, ds_a, ds_b])


2024: 2928 timesteps
2025: 2920 timesteps

Final dataset:
{'valid_time': 14608, 'latitude': 6, 'longitude': 7}
Time range: 2021-01-01T00:00:00.000000000 → 2025-12-31T21:00:00.000000000


C:\Users\udish\AppData\Local\Temp\ipykernel_23104\3920530274.py:23: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'latitude' ('latitude',) The recommendation is to set join explicitly for this case.
  merged = xr.merge([ds_i, ds_a, ds_b])
C:\Users\udish\AppData\Local\Temp\ipykernel_23104\3920530274.py:23: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  merged = xr.merge([ds_i, ds_a, ds_b])
C:\Users\udish\AppData\Local\Temp\ip

In [5]:
# ============================================================
# CELL 5 — Quick Validation
# ============================================================

print("Variables:", len(era5_full.data_vars))
print("BLH present:", "blh" in era5_full.data_vars)
print("Timesteps:", len(era5_full.valid_time))
print("Missing values:",
      int(sum(
          era5_full[v].isnull().sum().item()
          for v in era5_full.data_vars
      )))

Variables: 7
BLH present: True
Timesteps: 14608
Missing values: 224896


In [6]:
# ============================================================
# CELL 6 — Save Merged ERA5
# ============================================================

OUTPUT.parent.mkdir(parents=True, exist_ok=True)

era5_full.to_netcdf(OUTPUT)

print("Saved:", OUTPUT)

Saved: c:\Users\udish\College\MajorProjectSem7\Implementation\data\intermediate\era5_merged_2021_2025.nc
